# 02 — Preprocessing
Part of the *Kenya Household Food Security Classification* capstone project.

Feature engineering (Section 5), missingness screening (Section 6), and
preprocessing / train-test split / leakage-safe county encoding (Section 7).

**Input:** `data/processed/01_df.pkl` (from `01_eda.ipynb`)
**Output:** `data/processed/02_preprocessed.pkl` (consumed by `03_modelling.ipynb`)


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
path = "/content/drive/MyDrive/Ngao Labs Program/Capstone Project"


In [13]:
import pickle


# Construct the full path to the saved .pkl file
file_path_for_preprocessing = path + '/data/processed/01_df.pkl'

# Load the DataFrame from the .pkl file
with open(file_path_for_preprocessing, 'rb') as f:
    loaded_data_for_preprocessing = pickle.load(f)

df_from_pkl = loaded_data_for_preprocessing['df']

print(f"Successfully loaded DataFrame with shape: {df_from_pkl.shape} in this reference cell.")
display(df_from_pkl.head())

Successfully loaded DataFrame with shape: (4894, 2425) in this reference cell.


,hhid,wave,weight,weight_labor,weight_weekly,weight_knbs,weight_rdd,weight_panel_w1_2,weight_panel_w1_2_3,weight_panel_w1_2_3_4,...,head_education_level,s5_q39a_hungryadult_sev,s5_q39b_hungrychild_sev,s5_q40a_skippedadult_sev,s5_q40b_skippedchild_sev,s5_q41a_nofoodadult_sev,s5_q41b_nofoodchild_sev,fiss,food_security_status,hhs_score
0,3,4,101.287766,101.627686,41995.023438,537.997742,NaN,NaN,NaN,367.388489,...,2.0,0,0,1,0,0,0,1,mildly_insecure,0
1,4,4,77.351807,83.133888,4880.571289,96.752754,NaN,NaN,NaN,NaN,...,2.0,0,0,0,0,0,0,0,food_secure,0
2,5,4,197.629013,204.588501,29883.714844,483.506958,NaN,NaN,NaN,182.265030,...,2.0,0,0,0,0,0,0,1,mildly_insecure,0
3,6,4,1546.310547,1628.420166,29111.642578,5462.684570,NaN,NaN,NaN,NaN,...,4.0,0,0,0,0,0,0,0,food_secure,0
4,7,4,175.936142,182.131714,13904.761719,430.434509,NaN,NaN,NaN,107.680313,...,4.0,0,0,0,0,0,0,0,food_secure,0


## Setup

In [14]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42


## 5. Feature Engineering

Some features added in this round from the food-security column
review, each validated against the actual Wave 4 data (not assumed from the codebook)
before being included.

### 5.1 Wealth / asset index

`ownasset_curr_*` flags current ownership across 13 asset categories (0% missing).
Summing them gives an LSMS-style asset count: one of the most consistently strong
food-security predictors across country contexts.

In [16]:
df = df_from_pkl
asset_cols = [c for c in df.columns if c.startswith('ownasset_curr_')
              and c not in ['ownasset_curr__99', 'ownasset_curr__98']]
print(f"{len(asset_cols)} asset categories found")
df['n_assets_owned'] = df[asset_cols].sum(axis=1)
print(df['n_assets_owned'].describe())

13 asset categories found
count    4894.000000
mean        2.940539
std         1.032251
min         0.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         7.000000
Name: n_assets_owned, dtype: float64


### 5.2 Agriculture & livestock: structurally conditional, not missing-at-random

`s4_q4_acres` is 94.5% "missing," but that's because most sampled households simply
aren't farmers, 0/NaN means "not applicable," not "unknown." Mean-imputing acreage
across farmers and non-farmers alike would wash out a real distinction. Instead we add
an explicit `is_farming_hh` flag and fill the conditional fields to 0 for non-farmers.

In [17]:
df['is_farming_hh'] = df['s4_q4_acres'].notna().astype(int)
df['acres_if_farm'] = df['s4_q4_acres'].fillna(0)
df['livestock_value_filled'] = df['s4_q9_livestockvalue'].fillna(0)

print("Share of households that farm:", df['is_farming_hh'].mean().round(3))
print(df.groupby('is_farming_hh')['fiss'].mean())

Share of households that farm: 0.055
is_farming_hh
0    1.270703
1    1.505576
Name: fiss, dtype: float64


### 5.3 Enterprise activity, transfers, and expenditure

Income diversification (enterprises) and transfers (remittances, government, NGO aid)
are protective in FEWS NET's livelihoods framework; food-expenditure share is a classic
Engel's-law vulnerability signal — poorer/more food-insecure households spend a larger
share of total spending on food.

In [18]:
df['num_enterprises'] = df['s4_q12_num_ent'].fillna(0)

transfer_value_cols = ['s7_q1a_value_ken', 's7_q3a_value', 's7_q4a_value']  # remittance / govt / NGO
for c in transfer_value_cols:
    df[c + '_f'] = df[c].fillna(0)   # NaN = didn't receive that transfer type = 0 value
df['total_transfer_value'] = df[[c + '_f' for c in transfer_value_cols]].sum(axis=1)

exp_cols = ['s5_q3a_food', 's5_q3b_personal', 's5_q3c_durables', 's5_q3d_services', 's5_q3e_comms',
            's5_q3f_housing', 's5_q3g_utilities', 's5_q3h_transport', 's5_q3i_medical']
df['total_expenditure'] = df[exp_cols].sum(axis=1)
df['food_expenditure_share'] = np.where(
    df['total_expenditure'] > 0, df['s5_q3a_food'] / df['total_expenditure'], np.nan
)
df['food_expenditure_share'] = df['food_expenditure_share'].fillna(df['food_expenditure_share'].median())

print(df[['num_enterprises', 'total_transfer_value', 'food_expenditure_share']].describe())

       num_enterprises  total_transfer_value  food_expenditure_share
count      4894.000000           4894.000000             4894.000000
mean          0.046792            320.530854                0.422183
std           0.215050           1656.859780                0.213437
min           0.000000              0.000000                0.000000
25%           0.000000              0.000000                0.248007
50%           0.000000              0.000000                0.434783
75%           0.000000              0.000000                0.588235
max           2.000000          46000.000000                0.976744


### 5.4 Demographics: dependency ratio

`s2_q21_noadults` is 12.6% missing; filled with the training-distribution median since
no natural "0 means not applicable" logic applies here (every household has adults).

In [19]:
df['s2_q21_noadults_f'] = df['s2_q21_noadults'].fillna(df['s2_q21_noadults'].median())
df['dependency_ratio'] = (
    (df['s2_q15b_schoolchildren'].fillna(0) + df['s2_q15c_youngchildren'].fillna(0))
    / df['s2_q21_noadults_f'].replace(0, 1)
)
print(df['dependency_ratio'].describe())

count    4894.000000
mean        1.503426
std         1.593309
min         0.000000
25%         0.000000
50%         1.000000
75%         2.000000
max        15.000000
Name: dependency_ratio, dtype: float64


### 5.5 Distress-coping flags

The full `s6_copingactions_*` battery isn't available until Wave 5, but two individual
Wave-4 items capture the same underlying idea: selling assets under distress and taking
on debt. `s6_q1_soldasset` turns out to be a categorical code (1/2/3, occasionally a
combined "2 3"), not a clean binary, so it's kept as a categorical feature rather than
coerced into a flag it isn't.

In [20]:
print(df['s6_q1_soldasset'].value_counts(dropna=False))
print()
print(df['s6_q2_tookloan'].value_counts(dropna=False))

s6_q1_soldasset
1      4505
2       283
3       102
2 3       4
Name: count, dtype: int64

s6_q2_tookloan
0.0    4204
1.0     690
Name: count, dtype: int64


## 6. Missingness Screening

Same rule as before: from the full candidate pool, **drop any column still more than
70% missing** after the conditional 0/median fills in Section 5. Columns confirmed
structurally unavailable at Wave 4 (toilet type, water source, room count, the
`s6_copingactions_*` battery, broadband/mobile internet sub-flags, the original
household-level education field) are included in the pool specifically so this rule
demonstrably drops them, rather than being silently hand-picked out beforehand.

In [21]:
candidate_cols = {
    # demographics
    'urban': 'binary', 'hhsize': 'num', 'head_gender': 'cat', 'head_age': 'num',
    'head_age_3group': 'cat', 'strata': 'cat', 'any_adult_employed': 'binary',
    'hh_employment_rate': 'num', 'dependency_ratio': 'num', 'hh_avg_hours': 'num',
    'any_adult_laidoff': 'binary', 'any_migrant': 'binary',
    # education [NEW - corrected]
    'head_education_level': 'cat',
    's2_q29_hhheduc': 'cat',                 # expected to fail: 100% missing at W4
    # housing / infrastructure
    's2_q19_floormat': 'cat', 's2_q20_wallmat': 'cat',
    's2_q21a_powergrid': 'binary', 's2_hasinternet': 'binary',
    's2_typetoilet': 'cat', 's2_drinkwatersrc': 'cat', 's2_numhabrms': 'num',  # expected to fail
    's2_hasinternetbrdbnd': 'binary', 's2_hasinternetmbl': 'binary',            # expected to fail
    # wealth [NEW]
    'n_assets_owned': 'num',
    # agriculture [NEW]
    'is_farming_hh': 'binary', 'acres_if_farm': 'num', 'livestock_value_filled': 'num',
    # enterprise / transfers / expenditure [NEW]
    'num_enterprises': 'num', 'total_transfer_value': 'num', 'food_expenditure_share': 'num',
    's5_q4_rent': 'num',
    # coping [NEW]
    's6_q1_soldasset': 'cat', 's6_q2_tookloan': 'binary',
    's6_copingactions_1': 'binary', 's6_copingactions_2': 'binary', 's6_copingactions_3': 'binary',  # expected to fail
    # health access [NEW]
    's9_q4_hospitalvisit': 'binary', 's9_q11_medsoutstock': 'binary',
}

missing_after_fill = (df[list(candidate_cols.keys())].isnull().mean() * 100).sort_values(ascending=False)
print(missing_after_fill.round(2))

kept_cols = [c for c in candidate_cols if missing_after_fill[c] <= 70]
dropped_cols = [c for c in candidate_cols if missing_after_fill[c] > 70]
print("\nDropped (>70% missing):", dropped_cols)
print("\nKept:", len(kept_cols), "of", len(candidate_cols), "candidate columns")

s6_copingactions_2        100.00
s2_typetoilet             100.00
s2_hasinternetbrdbnd      100.00
s2_numhabrms              100.00
s2_drinkwatersrc          100.00
s2_q29_hhheduc            100.00
s6_copingactions_1        100.00
s2_hasinternetmbl         100.00
s6_copingactions_3        100.00
head_education_level        0.22
s2_q20_wallmat              0.04
s5_q4_rent                  0.04
s9_q11_medsoutstock         0.02
s2_q19_floormat             0.02
strata                      0.00
head_age_3group             0.00
head_age                    0.00
head_gender                 0.00
hhsize                      0.00
urban                       0.00
any_adult_laidoff           0.00
any_migrant                 0.00
s2_q21a_powergrid           0.00
s2_hasinternet              0.00
any_adult_employed          0.00
hh_employment_rate          0.00
hh_avg_hours                0.00
dependency_ratio            0.00
is_farming_hh               0.00
n_assets_owned              0.00
food_expen

As expected, `s2_q29_hhheduc`, `s2_typetoilet`, `s2_drinkwatersrc`, `s2_numhabrms`,
the `s6_copingactions_*` battery, and the broadband/mobile internet sub-flags are all
dropped — confirming they're structurally unavailable at Wave 4 rather than something
worth imputing around. Everything else survives, including the newly added wealth,
agriculture, enterprise, transfer, expenditure, and coping features.

## 7. Preprocessing

### 7.1 Feature Encoding

Categorical features are one-hot encoded, binary features used as-is, numeric features
used directly (tree models don't need scaling). `s2_q9a_county` is excluded from the
one-hot set — with 47 categories it would fragment into 47 sparse columns; instead it
gets a leakage-safe target encoding in 7.3, computed after the train/test split.

In [22]:
cat_cols = [c for c, t in candidate_cols.items() if t == 'cat' and c in kept_cols]
bin_cols = [c for c, t in candidate_cols.items() if t == 'binary' and c in kept_cols]
num_cols = [c for c, t in candidate_cols.items() if t == 'num' and c in kept_cols]

# remaining missingness in kept numeric/binary columns (all <=70% but a few still nonzero) -> fill
for c in num_cols:
    if df[c].isnull().any():
        df[c] = df[c].fillna(df[c].median())
for c in bin_cols:
    if df[c].isnull().any():
        df[c] = df[c].fillna(0)

print("Categorical (one-hot):", cat_cols)
print("Binary:", bin_cols)
print("Numeric:", num_cols)

X_cat = pd.get_dummies(df[cat_cols].astype('category'), prefix=cat_cols)
X = pd.concat([df[bin_cols + num_cols].reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)

y_labels = df['food_security_status']
le = LabelEncoder()
y = le.fit_transform(y_labels)
print("\nClasses:", list(le.classes_))
print("Feature matrix shape (before county encoding):", X.shape)
X.head()

Categorical (one-hot): ['head_gender', 'head_age_3group', 'strata', 'head_education_level', 's2_q19_floormat', 's2_q20_wallmat', 's6_q1_soldasset']
Binary: ['urban', 'any_adult_employed', 'any_adult_laidoff', 'any_migrant', 's2_q21a_powergrid', 's2_hasinternet', 'is_farming_hh', 's6_q2_tookloan', 's9_q4_hospitalvisit', 's9_q11_medsoutstock']
Numeric: ['hhsize', 'head_age', 'hh_employment_rate', 'dependency_ratio', 'hh_avg_hours', 'n_assets_owned', 'acres_if_farm', 'livestock_value_filled', 'num_enterprises', 'total_transfer_value', 'food_expenditure_share', 's5_q4_rent']

Classes: ['food_secure', 'mildly_insecure', 'moderately_insecure', 'severely_insecure']
Feature matrix shape (before county encoding): (4894, 67)


,urban,any_adult_employed,any_adult_laidoff,any_migrant,s2_q21a_powergrid,s2_hasinternet,is_farming_hh,s6_q2_tookloan,s9_q4_hospitalvisit,s9_q11_medsoutstock,...,s2_q20_wallmat_13.0,s2_q20_wallmat_14.0,s2_q20_wallmat_15.0,s2_q20_wallmat_16.0,s2_q20_wallmat_17.0,s2_q20_wallmat_18.0,s6_q1_soldasset_1,s6_q1_soldasset_2,s6_q1_soldasset_2 3,s6_q1_soldasset_3
0,1,1.0,0.0,0.0,1,1.0,0,0.0,0.0,2.0,...,False,False,True,False,False,False,True,False,False,False
1,1,1.0,0.0,3.0,0,0.0,0,0.0,0.0,0.0,...,False,False,False,False,False,False,True,False,False,False
2,1,1.0,0.0,3.0,0,0.0,0,0.0,0.0,0.0,...,False,False,False,False,False,False,True,False,False,False
3,0,0.0,0.0,3.0,1,0.0,0,0.0,0.0,0.0,...,False,False,False,False,False,False,True,False,False,False
4,1,0.0,0.0,3.0,1,1.0,0,0.0,1.0,0.0,...,False,False,False,False,False,False,True,False,False,False


### 7.2 Train-Test Split

Stratified by the target so each of the four classes keeps its proportion in both
splits. `urban_series` and `county_series` ride along unencoded for disaggregated
evaluation later, and so `county_series` is available for the target encoding below.

In [23]:
urban_series = df['urban'].reset_index(drop=True)
county_series = df['s2_q9a_county'].reset_index(drop=True)

X_train, X_test, y_train, y_test, urban_train, urban_test, county_train, county_test = train_test_split(
    X, y, urban_series, county_series,
    test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train size:", X_train.shape[0], " Test size:", X_test.shape[0])

Train size: 3915  Test size: 979


### 7.3 County Target Encoding (leakage-safe)

Kenya's food security varies sharply by county (drought-prone ASAL counties vs.
high-potential agricultural zones). Rather than one-hot encoding 47 sparse columns, we
compute each county's food-insecurity rate — **using only the training set's labels** —
and map it onto both splits as a single numeric feature. Counties in the test set but
not the training set fall back to the overall training insecurity rate.

In [24]:
secure_code = le.transform(['food_secure'])[0]
insecure_train = (y_train != secure_code).astype(int)

county_rate_map = pd.Series(insecure_train, index=county_train.values).groupby(level=0).mean()
global_insecurity_rate = insecure_train.mean()

X_train = X_train.copy()
X_test = X_test.copy()
X_train['county_insecurity_rate'] = county_train.map(county_rate_map).values
X_test['county_insecurity_rate'] = county_test.map(county_rate_map).fillna(global_insecurity_rate).values

print("Final feature matrix shape:", X_train.shape)
print("\nCounty insecurity rate — top 5 highest-risk counties in training data:")
print(county_rate_map.sort_values(ascending=False).head())

Final feature matrix shape: (3915, 68)

County insecurity rate — top 5 highest-risk counties in training data:
40    0.643564
25    0.642857
23    0.625000
7     0.608696
36    0.605263
dtype: float64


## 14. Save Output for the Next Notebook

In [26]:
import os
os.makedirs(path + '../data/processed', exist_ok=True)
with open(path + '../data/processed/02_preprocessed.pkl', 'wb') as f:
    pickle.dump({
        'df': df,
        'candidate_cols': candidate_cols, 'kept_cols': kept_cols, 'dropped_cols': dropped_cols,
        'cat_cols': cat_cols, 'bin_cols': bin_cols, 'num_cols': num_cols,
        'asset_cols': asset_cols, 'exp_cols': exp_cols, 'transfer_value_cols': transfer_value_cols,
        'X': X, 'y': y, 'le': le,
        'X_train': X_train, 'X_test': X_test, 'y_train': y_train, 'y_test': y_test,
        'urban_train': urban_train, 'urban_test': urban_test,
        'county_train': county_train, 'county_test': county_test,
        'county_rate_map': county_rate_map, 'global_insecurity_rate': global_insecurity_rate,
        'RANDOM_STATE': RANDOM_STATE,
    }, f)
print('Saved preprocessing artifacts to ../data/processed/02_preprocessed.pkl')
print('X_train:', X_train.shape, ' X_test:', X_test.shape)


Saved preprocessing artifacts to ../data/processed/02_preprocessed.pkl
X_train: (3915, 68)  X_test: (979, 68)
